(course-core-covalent-connectivity)=

# Module 15: Covalent Connectivity

Molecular systems are held together by covalent bonds. Understanding this connectivity is essential for defining groups, components, and molecules. 

In this module, we will explore how MolSysMT handles bonds and how we can reconstruct them if they are missing.

In [ ]:
import molsysmt as msm
from molsysmt import systems
import networkx as nx

lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

### 1. Extracting Covalent Bonds
The most basic way to see the connectivity is asking for the `bonded_atom_pairs`. This returns an array where each row is a pair of indices of bonded atoms.

In [ ]:
bonds = msm.get(lysozyme, element='system', bonded_atom_pairs=True)

print(f"Total covalent bonds in the system: {len(bonds)}")
print(f"First 5 bonds: \n{bonds[:5]}")

### 2. Topological Analysis with `msm.topology` 
MolSysMT can group atoms into covalently connected sets. This is how we distinguish a protein chain from a ligand or a water molecule.

- `get_covalent_blocks()`: Returns sets of atoms that are covalently connected to each other.

In [ ]:
blocks = msm.topology.get_covalent_blocks(lysozyme)
print(f"Found {len(blocks)} independent covalent blocks in the system.")
print(f"Size of the first block (protein): {len(blocks[0])} atoms.")

### 3. Selection by Connectivity
You can select atoms based on their bonds using the `bonded to` operator in the selection language.

In [ ]:
# Select all atoms bonded to the atom with index 10
bonded_to_10 = msm.select(lysozyme, selection='bonded to atom_index==10')

print(f"Atoms covalently bonded to atom 10: {bonded_to_10}")

### 4. Reconstructing Missing Bonds (Inference)
If your input file (like a raw PDB) does not contain a full list of bonds, you can tell MolSysMT to reconstruct them based on chemical templates or distances between atoms.

In [ ]:
raw_pdb = 'pdb:1VII'
print(f"Has bonds initially? {msm.has_attribute(raw_pdb, 'bonded_atom_pairs')}")

# We convert and add missing bonds
molsys = msm.convert(raw_pdb, to_form='molsysmt.MolSys')
msm.build.add_missing_bonds(molsys)

print(f"Number of bonds after reconstruction: {msm.get(molsys, element='system', n_bonds=True)}")

### 5. Connectivity as a Graph
For complex topological queries (like finding rings or calculating the diameter of a molecule), we can convert the system into a **NetworkX Graph**.

In [ ]:
# Convert the connectivity into a NetworkX Graph form
graph = msm.convert(lysozyme, to_form='networkx.Graph')

print(f"Graph created with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")

--- 

### 🏆 Challenge 12: The Bond Detective

1. Load the **SARS-CoV-2 Protease** from its PDB ID.
2. Use `msm.build.add_missing_bonds()` to ensure its connectivity is complete.
3. Use `msm.select()` to find which atoms are bonded to the first Sulfur atom of the system.
4. Convert the system to a graph and use NetworkX to find the number of **connected components**.